**Make sure you load the API keys for cloud providers!**

You can set your environment keys yourself or use a script. Please note that since keys are private, they are not included in the repository.

In [2]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from config import set_environment
# for the keys - as explained early in chapter 2
set_environment()

# Basic RAG Implementation

In [4]:
# For query trnasformation
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# For basic RAG implementation
from langchain_community.document_loaders import JSONLoader
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load documents
loader = JSONLoader(
    file_path="knowledge_base.json",
    jq_schema=".[].content",
    text_content=True, 
)
documents = loader.load()

# 2. Convert to vectors
embedder = OpenAIEmbeddings(model="text-embedding-3-large")
embeddings = embedder.embed_documents([doc.page_content for doc in documents])

# 3. Store in vector database
vector_db = FAISS.from_documents(documents, embedder)

# # 4. Retrieve similar docs
query = "What are the effects of climate change?"
results = vector_db.similarity_search(query)

In [ ]:
print(len(results))
print(results)

4
[Document(id='9a3a7cff-d581-4d3e-95ec-a0eebc74ea7f', metadata={'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 3}, page_content='GPT (Generative Pre-trained Transformer) models are autoregressive language models that use transformer-based neural networks. Unlike BERT, which is bidirectional, GPT models are unidirectional and predict the next token based on previous tokens. The original GPT was introduced by OpenAI in 2018, followed by GPT-2 in 2019 and GPT-3 in 2020, each significantly larger than its predecessor.'), Document(id='15f9d46b-bd58-409d-920e-7ab98bc465d8', metadata={'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 4}, page_content='Retrieval-Augmented Generation (RAG) combines a retrieval system with a text generator. The retriever fetches relevant documents from a knowledge base, and these documents are then provided as context to the gen

'GPT (Generative Pre-trained Transformer) models are autoregressive language models that use transformer-based neural networks. Unlike BERT, which is bidirectional, GPT models are unidirectional and predict the next token based on previous tokens. The original GPT was introduced by OpenAI in 2018, followed by GPT-2 in 2019 and GPT-3 in 2020, each significantly larger than its predecessor.'

# KNN Retriever

In [8]:
from langchain_community.retrievers import KNNRetriever
from langchain_openai import OpenAIEmbeddings

retriver = KNNRetriever.from_documents(documents, OpenAIEmbeddings())
results = retriver.invoke("query")

In [10]:
results

[Document(metadata={'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 4}, page_content='Retrieval-Augmented Generation (RAG) combines a retrieval system with a text generator. The retriever fetches relevant documents from a knowledge base, and these documents are then provided as context to the generator. RAG models can be fine-tuned end-to-end and leverage large pre-trained models like BART or T5 for generation. This approach helps ground the generated text in factual information.'),
 Document(metadata={'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 1}, page_content="Transformer models were introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. The architecture relies on self-attention mechanisms rather than recurrent or convolutional neural networks. This design allows for more parallelization during training and better handlin

# External Search API Retriever

In [13]:
from langchain_community.retrievers.pubmed import PubMedRetriever

retriever = PubMedRetriever()
results = retriever.invoke("COVID research")

In [15]:
results

[Document(metadata={'uid': '41051810', 'Title': '[Effectiveness and tolerability of agomelatine (Valdoxan) in the treatment of depression after COVID-19 (TELESFOR study)].', 'Published': '--', 'Copyright Information': ''}, page_content='OBJECTIVE: To study the antidepressant efficacy and the tolerability of agomelatine (Valdoxan) in daily clinical practice in patients with a depressive episode that occurred after COVID-19.\nMATERIAL AND METHODS: The final analysis included 103 patients (70.9% women, average age 45 years). Of these, 78.6% have had mild COVID-19 in the last 3 months, and 21.4% have had moderate COVID-19. The average time from the onset of infection to inclusion in the study was 2.1±0.7 months. At the start of the study, 53.4% of patients had a mild depressive episode, and 46.6% had a moderate depressive episode. Clinical and psychopathological, psychometric (HAMD-17, CGI, SF-36), and statistical methods of investigation were used.\nRESULTS: A significant decrease in the 